# Day2：LoRAの学習量とFAQの変更を比較する

講座配布の `notebooks/lora_format_tuning.ipynb`（v1.1）を基にした再実行用ノートブックです。
実測値と考察は `answer.md`、出力原文・データ・ログは `results/` に保存しています。
ノートブックの保存済み出力は空にしてあり、結果の出所は条件別のJSONで確認できます。

## 実行方法

1. Google Colabにこのファイルをアップロードし、GPUランタイムを選びます。
2. 下の `RUN` を `baseline`・`basic_epochs10`・`implementation` から選びます。
3. **ランタイム → セッションを再起動してすべて実行する** を選びます。学習済みモデルへの追加学習を避けるため、条件を変えるたびに再起動します。
4. 最後に作られるZIPをダウンロードします。別条件のファイルは別フォルダに保存されます。
5. 必要なファイルを保存してから、ランタイムを接続解除して削除します。

今回の実測はNVIDIA L4で行いました。T4の割り当てを試みましたが利用できず、3条件をL4で統一しました。
公開モデルなのでHugging FaceのトークンやGoogle Driveのマウントは不要です。


## 0. 環境

GPUを確認し、講座指定のライブラリを導入します。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 比較条件

基礎は20→10エポックのみ変更。実装は標準の20エポックに戻し、FAQと質問を変更します。乱数seed・モデルrevision・生成方法・その他の学習設定は共通です。

In [ ]:
RUN = 'baseline'  # baseline / basic_epochs10 / implementation
assert RUN in {'baseline', 'basic_epochs10', 'implementation'}
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 10 if RUN == "basic_epochs10" else 20,  # 変更する学習設定はこの1つだけ
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": f"/content/day2_homework/{RUN}",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))
from datetime import datetime, timezone
RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat()
print("実験条件:", RUN)


## 2. FAQとデータ分割

実装では架空の地域AI学習会の5項目を用います。train=14、valid=7、test=21。質問文の重複を禁止し、test専用の言い換えも用意します。金額・場所などはすべて架空です。

In [ ]:
if RUN == "implementation":
    # 架空の地域AI学習会。実在の募集条件や個人情報ではありません。
    FAQ = {
        "schedule": "学習会は毎週土曜 10:00〜12:00 に開催します。",
        "venue": "会場は地域交流センターの第2会議室です。",
        "fee": "参加費は 1 回 500 円で、当日受付で現金払いです。",
        "items": "持ち物はノートパソコンと充電器です。",
        "capacity": "定員は各回 12 名で、申込フォームから先着順で受け付けます。",
    }
    FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())
    GROUNDED = {
        "schedule": (["学習会はいつ開催されますか", "開催曜日と時間を教えてください", "何曜日の何時に集まりますか", "学習会の開催時間は？"], ["土曜", "10:00", "12:00"]),
        "venue": (["会場はどこですか", "学習会の場所を教えてください", "どの施設のどの部屋に行けばよいですか", "集まる場所は？"], ["地域交流センター", "第2会議室"]),
        "fee": (["参加費はいくらでどう払いますか", "料金と支払方法を教えてください", "費用と支払うタイミングは？", "参加料金の金額と払い方は？"], ["500", "当日", "現金"]),
        "items": (["何を持参すればよいですか", "必要な持ち物を教えてください", "準備するものは何ですか", "当日の持参品は？"], ["ノートパソコン", "充電器"]),
        "capacity": (["何人まで参加できてどう申し込みますか", "定員と申込方法を教えてください", "参加枠の人数と受付方法は？", "人数制限と申込の手順は？"], ["12", "申込フォーム", "先着順"]),
    }
    # FAQの項目に近いが答えが書かれていない質問を含める。
    UNGROUNDED = [
        "参加に年齢制限はありますか", "会場に駐車場はありますか", "初心者向けの予習教材はありますか", "欠席した場合に録画を見られますか",
        "パソコンの貸し出しはありますか", "キャンセルしたら返金されますか", "会場でWi-Fiを利用できますか", "オンラインでも参加できますか",
    ]
    TEST_EXTRA_GROUNDED = {
        "schedule": ["学習会の曜日と開始・終了時刻を知りたいです", "何曜日の何時から何時まで行いますか"],
        "venue": ["集合する施設名と部屋名を教えてください", "学習会はどこに行けば参加できますか"],
        "fee": ["1回分の料金と支払う時期・方法を知りたいです", "当日の参加費と決済方法は何ですか"],
        "items": ["参加時に用意して持っていくものを教えてください", "自分で持ってくる機器は何ですか"],
        "capacity": ["各回の募集人数と参加申請の方法を知りたいです", "申込は何名まで、どの窓口で受け付けますか"],
    }
    TEST_EXTRA_UNGROUNDED = [
        "参加費に学生割引はありますか", "定員を超えたらキャンセル待ちはできますか",
        "持参するパソコンのOSに指定はありますか", "会場は車いすで利用できますか",
    ]
else:
    FAQ = {
        "deadline": "課題の提出期限は 9 月 30 日 23:59（日本時間）です。",
        "destination": "提出先は学習ポータルの「課題」欄です。",
        "format": "提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。",
        "office_hours": "質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。",
        "late": "期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。",
    }
    FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())

    # 根拠あり: (FAQ キー, 質問テンプレート群, 正解の answer に含まれるべき語)
    GROUNDED = {
        "deadline": (["課題の提出期限はいつですか", "いつまでに出せばいいですか", "締切を教えてください", "提出の期限は？"], ["9 月 30 日", "23:59"]),
        "destination": (["提出先はどこですか", "どこにアップロードしますか", "課題はどこから出しますか", "提出場所は？"], ["学習ポータル", "課題"]),
        "format": (["提出形式を教えてください", "ファイル名の決まりはありますか", "何のファイル形式で出しますか", "PDF 以外でも出せますか"], ["PDF"]),
        "office_hours": (["質問はいつできますか", "オフィスアワーはいつですか", "先生に質問する時間はありますか", "相談できる時間帯は？"], ["火曜", "17:00"]),
        "late": (["遅れて出したらどうなりますか", "期限を過ぎた場合は受け付けますか", "遅延提出のルールは？", "締切後の扱いを教えてください"], ["48 時間", "減点"]),
    }
    # 根拠なし: FAQ に無いので needs_review=true が正解
    UNGROUNDED = [
        "再提出はできますか", "欠席した回の扱いを教えてください", "他の受講者の点数は見られますか",
        "成績はいつ出ますか", "提出期限は延長できますか", "グループでの提出は可能ですか",
        "課題の配点はどうなっていますか", "講義の録画は公開されますか",
    ]

    # test 専用の追加質問（train / valid には入れない）
    TEST_EXTRA_GROUNDED = {
        "deadline": ["提出期限の日付と時刻を教えてください", "課題の締切は何月何日の何時ですか"],
        "destination": ["課題の提出はどこで行いますか", "提出物を出す先を教えてください"],
        "format": ["提出するファイルの形式は何ですか", "提出ファイルは何形式で、いくつ出しますか"],
        "office_hours": ["オフィスアワーの曜日と時間を教えてください", "質問を受け付けてもらえるのは何曜日の何時からですか"],
        "late": ["提出が締切に間に合わなかったらどうなりますか", "期限後に提出した課題は受理されますか"],
    }
    TEST_EXTRA_UNGROUNDED = [
        "課題の文字数に制限はありますか", "提出した課題へのフィードバックはもらえますか",
        "参考文献の書き方に指定はありますか", "オフィスアワーは事前予約が必要ですか",
    ]

UNGROUNDED_ANSWER = "この FAQ には記載がありません。運営に確認が必要です。"

def make_example(question: str, answer: str, needs_review: bool) -> dict:
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "faq": FAQ_TEXT, "target": target, "needs_review": needs_review}

def split_templates(templates):
    # テンプレートの先頭 2 つを train、3 つ目を valid、4 つ目を test に固定（言い換えを跨がせない）
    return templates[:2], templates[2:3], templates[3:4]

train, valid, test, gold = [], [], [], {}
for key, (templates, must_include) in GROUNDED.items():
    tr, va, te = split_templates(templates)
    for q in tr: train.append(make_example(q, FAQ[key], False))
    for q in va: valid.append(make_example(q, FAQ[key], False))
    for q in te + TEST_EXTRA_GROUNDED[key]:
        test.append(make_example(q, FAQ[key], False)); gold[q] = {"needs_review": False, "must_include": must_include}
for i, q in enumerate(UNGROUNDED):
    ex = make_example(q, UNGROUNDED_ANSWER, True)
    bucket = [train, train, valid, test][i % 4]   # 根拠なしの質問は 4 問ごとに train 2 / valid 1 / test 1 に振る
    bucket.append(ex)
    if bucket is test: gold[q] = {"needs_review": True, "must_include": ["確認"]}
for q in TEST_EXTRA_UNGROUNDED:
    test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {"needs_review": True, "must_include": ["確認"]}

# train は 14 件（根拠あり 10、根拠なし 4）。同じ例を何回学習するかは、データを複製せず、セクション 4 の num_train_epochs で決める
all_questions = [r["question"] for r in train + valid + test]
assert len(set(all_questions)) == len(all_questions), "質問が train / valid / test のどこかで重複しています。テンプレートを見直してください"
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
print("学習例の 1 件 | 質問:", train[0]["question"], "| 望ましい出力:", train[0]["target"])
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")
(WORK / "valid.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in valid), encoding="utf-8")


## 3. モデルと学習前評価

FAQと質問のみを入力し、正解は採点時だけ参照します。講座の形式判定・内容判定を使用します。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# 形式はプロンプトで指示している。モデルがこれを守れるかが学習前の確認点
SYSTEM = "あなたは講座 FAQ の回答係です。与えられた FAQ だけを根拠に、必ず JSON 1 行で答えてください。形式: {\"answer\": 文字列, \"needs_review\": 真偽値}。FAQ に記載が無いことは answer に『確認が必要』と書き、needs_review を true にします。"

def build_messages(example):
    user = f"FAQ:\n{example['faq']}\n\n質問: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 出力の原文を省略せずに表示する

def check_format(text: str):
    '''JSON として読め、必須キーと型が合うか。失敗理由も返す'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''内容の合否: needs_review が正解と一致し、必須語句を answer に含む'''
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str):
    '''test の 21 問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # 形式が壊れていても JSON らしい部分が読めるなら、内容だけは別に採点する
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# 質問ごとの出力。「正解の needs_review」と見比べ、どのキーが欠けているか、文面は FAQ と合っているかを読む
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "学習前の出力": r["raw"],
                       "形式": mark(r["format_ok"]), "形式の失敗理由": r["format_reason"], "内容": mark(r["content_ok"])} for r in before_rows]))

## 4. LoRA学習

重みはfloat32、量子化なし。L4でも混合精度が自動で有効にならないよう、fp16とbf16を両方Falseに固定します。最終lossは最終10ステップ区間のログ値であり、検証損失ではありません。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # 学習データは「入力（system + user）」と「望ましい出力（assistant）」の対。評価と同じ build_messages を使う
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, bf16=False, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
train_output = trainer.train()  # 途中で止まった場合の再開は 4b を参照
training_seconds = time.time() - t0
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）
loss_logs = [row for row in trainer.state.log_history if "loss" in row]
training_record = {
    "seconds": training_seconds,
    "final_logged_loss": loss_logs[-1]["loss"],
    "final_loss_step": loss_logs[-1]["step"],
    "mean_training_loss": train_output.training_loss,
    "global_step": trainer.state.global_step,
    "epoch": trainer.state.epoch,
    "fp16": sft.fp16, "bf16": sft.bf16,
    "trainable_parameters": tuned.get_nb_trainable_parameters()[0],
    "all_parameters": tuned.get_nb_trainable_parameters()[1],
    "logs": trainer.state.log_history,
}
(WORK / "training.json").write_text(json.dumps(training_record, ensure_ascii=False, indent=2), encoding="utf-8")


## 5. 学習後評価

学習前と同じ21問、同じ生成条件で評価し、退行した質問も確認します。

In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# 学習前は形式も内容も通ったのに、学習後に落ちた質問（退行）
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

# 質問ごとの出力原文。形式（JSON・必須キー・型）と内容（needs_review の一致と必須語句）の判定を自分の目で確かめる
detail = pd.DataFrame([{"質問": b["question"], "正解の needs_review": b["gold_needs_review"],
                        "学習前の出力": b["raw"], "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

## 6. 未記載の事実を尋ねる追加4問

未知の質問3問と、FAQ内の対照質問1問です。対照質問の有無はtestとの完全一致ではなく、FAQに実際に記載があるかで表示します。

In [ ]:
if RUN == "implementation":
    KNOWLEDGE_QUESTIONS = [
        "この学習会の主催者の名前は何ですか",
        "来月の特別講座の開催日はいつですか",
        "学習会で資格を取得できますか",
        "学習会の開催時間は？",
    ]
else:
    KNOWLEDGE_QUESTIONS = [
        "この講座の最終課題の締切はいつですか",       # FAQ に無い（架空）
        "担当教員の名前は何ですか",                     # FAQ に無い
        "次回の講義のテーマは何ですか",                 # FAQ に無い
        "課題の提出期限はいつですか",                   # FAQ にある（対照）
    ]

def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

knowledge_rows = []
for q in KNOWLEDGE_QUESTIONS:
    ex = make_example(q, "", True)
    # LoRA は基盤モデルの層に差し込まれるので、学習後は `model` で生成しても adapter 込みになる。学習前の出力は adapter を一時的に無効にして取る
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"質問": q, "FAQ にある": q == KNOWLEDGE_QUESTIONS[-1], "学習前の分類": classify(before_raw), "学習前の出力": before_raw,
                 "学習後の分類": classify(after_raw), "学習後の出力": after_raw})
display(pd.DataFrame(knowledge_rows))

## 7. 記録の保存

学習時間、損失ログ、GPU名、実行日時、設定、学習前後のスコアを記録します。講師参考値ではなく、このランタイムで学習を実行した結果を保存します。

In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "run": RUN,
    "started_utc": RUN_STARTED_UTC,
    "completed_utc": datetime.now(timezone.utc).isoformat(),
    "gpu": torch.cuda.get_device_name(0),
    "training": training_record,
    "dataset_sizes": {"train": len(train), "valid": len(valid), "test": len(test)},
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # 講師の結果を使った場合は False にする
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
archive = Path("/content") / f"day2_{RUN}.zip"
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for name in ["result.json", "training.json", "eval_before.json", "eval_after.json", "train.jsonl", "valid.jsonl", "test.jsonl", "gold.json"]:
        z.write(WORK / name, arcname=f"{RUN}/{name}")
print("保存完了:", archive)


## 8. ダウンロード（任意）

上で生成された8ファイルをまとめたZIPです。モデル重み・APIキー・個人情報は含みません。

In [ ]:
from google.colab import files
files.download(str(archive))
